# Lesson 07 - การออกแบบรูปแบบการวางแผน

สมุดบันทึกนี้แสดงตัวอย่าง **รูปแบบการวางแผนการออกแบบ** สำหรับเอเจนต์ AI โดยใช้ Microsoft Agent Framework
คุณจะได้เรียนรู้วิธีการแยกคำขอการเดินทางที่ซับซ้อนออกเป็นงานย่อยที่มีโครงสร้าง กำหนดงานเหล่านั้นให้กับเอเจนต์ผู้เชี่ยวชาญ
และดำเนินการแผนที่ได้ผลลัพธ์ — ทั้งหมดนี้ใช้ผลลัพธ์ที่มีโครงสร้างโดยใช้โมเดล Pydantic


## การตั้งค่า


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## การแยกงานออกเป็นส่วนย่อย

การแยกงานออกเป็นส่วนย่อยคือหัวใจของรูปแบบการออกแบบการวางแผน แทนที่จะขอให้ตัวแทนเดียวจัดการคำขอที่ซับซ้อนทั้งหมดตั้งแต่ต้นจนจบ เราจะแบ่งปัญหาออกเป็น **งานย่อย** ที่มีการกำหนดอย่างชัดเจนแต่ละงานย่อยจะถูกมอบหมายให้กับตัวแทนผู้เชี่ยวชาญเฉพาะด้าน (เช่น เที่ยวบิน, โรงแรม, กิจกรรม) โดยมีลำดับความสำคัญและลำดับความสัมพันธ์ที่ชัดเจน

วิธีนี้ให้ประโยชน์หลายประการ:
- **ความชัดเจน**: งานย่อยแต่ละงานมีความรับผิดชอบเพียงอย่างเดียว
- **ความขนาน**: งานย่อยที่ไม่ขึ้นต่อกันสามารถทำงานพร้อมกันได้
- **ความน่าเชื่อถือ**: ความล้มเหลวจะถูกแยกไว้ภายในงานย่อยแต่ละตน
- **การติดตามงบประมาณ**: ค่าใช้จ่ายถูกประมาณสำหรับแต่ละงานย่อยและสรุปรวม


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## การสร้างตัวแทนวางแผนด้วย Structured Output

ตัวแทนวางแผนทำหน้าที่เป็น **พนักงานต้อนรับหน้าเคาน์เตอร์** เมื่อได้รับคำขอเดินทางระดับสูง ตัวแทนจะสร้าง `TravelPlan` ที่มีโครงสร้าง — แยกคำขอนั้นออกเป็นงานย่อย กำหนดลำดับความสำคัญ และระบุความสัมพันธ์เพื่อให้พนักงานคอนเซียร์จหรือชั้นการดำเนินงานสามารถทำงานให้เสร็จได้


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## การดำเนินแผนงานด้วยเครื่องมือเฉพาะทาง

เมื่อเจ้าหน้าที่เคาน์เตอร์หน้าได้รับแผนงานที่มีโครงสร้างแล้ว **เจ้าหน้าที่ผู้ช่วยส่วนตัว** จะดำเนินการตามแผนนั้น
เครื่องมือเฉพาะทางแต่ละตัวจะจัดการกับประเภทของงานย่อยหนึ่งประเภท (เที่ยวบิน โรงแรม กิจกรรม) เจ้าหน้าที่ผู้ช่วยส่วนตัวจะทำซ้ำผ่านงานย่อยในแผนตามลำดับความขึ้นต่อกันและส่งแต่ละงานไปยังเครื่องมือที่เหมาะสมต่อไป


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้เกี่ยวกับ **รูปแบบการออกแบบวางแผน (Planning Design Pattern)** สำหรับตัวแทน AI:

1. **การแยกย่อยงาน (Task Decomposition)** — ตัวแทนวางแผนที่แผนกต้อนรับจะแยกคำขอการเดินทางที่ซับซ้อนออกเป็นงานย่อยที่มีโครงสร้างโดยใช้โมเดล Pydantic และมอบหมายแต่ละงานให้กับตัวแทนผู้เชี่ยวชาญพร้อมลำดับความสำคัญและการพึ่งพา
2. **ผลลัพธ์ที่มีโครงสร้าง (Structured Output)** — โดยการส่งผ่าน `response_format` ตัวแทนจะส่งกลับวัตถุ `TravelPlan` ที่ได้รับการตรวจสอบแทนการส่งข้อความแบบอิสระ ช่วยให้การประมวลผลต่อเนื่องน่าเชื่อถือ
3. **การดำเนินแผน (Plan Execution)** — ตัวแทนผู้ช่วยคอนเซียร์จจะทำงานผ่านงานย่อยโดยใช้เครื่องมือผู้เชี่ยวชาญ (`book_flight`, `reserve_hotel`, `book_activity`) เพื่อดำเนินแผนและรายงานผลลัพธ์

รูปแบบนี้แยกความแตกต่างระหว่าง *สิ่งที่ต้องทำ* (การวางแผน) กับ *วิธีการทำ* (การดำเนินงาน) ทำให้ตัวแทนมีความยืดหยุ่น ทดสอบง่าย และขยายเพิ่มเติมได้ง่ายขึ้น


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้ความถูกต้องเป็นอย่างสูง แต่กรุณาทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือตำหนิได้ เอกสารต้นฉบับในภาษาดั้งเดิมถือเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่มีความสำคัญ แนะนำให้ใช้การแปลโดยผู้เชี่ยวชาญมืออาชีพ เราจะไม่รับผิดชอบต่อความเข้าใจผิดหรือความคลาดเคลื่อนที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
